# Simulation Analysis - Live Data

This notebook reads the SQLite database populated by `make sim` and
visualises the monitoring pipeline's behaviour over real recorded batches.

**Prerequisite:** run `make train && make sim` before opening this notebook.
The notebook reads from `data/metrics/metrics.db` which is populated by the
simulation loop.

For a self-contained walkthrough that requires no prior data, see
[`drift_simulation.ipynb`](drift_simulation.ipynb) instead.

---

## Setup

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

DB = Path("../data/metrics/metrics.db")
assert DB.exists(), f"Database not found at {DB}. Run 'make train && make sim' first."

plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#333355",
    "axes.labelcolor":  "#ccccdd",
    "xtick.color":      "#ccccdd",
    "ytick.color":      "#ccccdd",
    "text.color":       "#eeeeff",
    "grid.color":       "#222244",
})

## Load monitoring records

In [ ]:
con = sqlite3.connect(DB)
df = pd.read_sql(
    "SELECT * FROM metrics_records ORDER BY timestamp",
    con,
    parse_dates={"timestamp": {"unit": "s"}},
)
con.close()

print(f"{len(df)} batches recorded")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")
df[["accuracy", "f1", "drift_score", "decision_latency_ms"]].describe().round(4)

## Performance & drift over time

Vertical lines mark decision events (retrain, rollback, promote, reject).

In [ ]:
ACTION_COLOURS = {
    "reject":   "#e74c3c",
    "rollback": "#e67e22",
    "retrain":  "#f39c12",
    "promote":  "#2ecc71",
}

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.patch.set_facecolor("#0f1117")

for ax in axes:
    ax.set_facecolor("#1a1a2e")
    ax.spines[:].set_color("#333355")

# Panel 1: F1 and accuracy
axes[0].plot(df["timestamp"], df["accuracy"], label="accuracy", color="#80cbc4", lw=1.5)
axes[0].plot(df["timestamp"], df["f1"],       label="F1",       color="#b39ddb", lw=1.5)
axes[0].axhline(0.75, color="#e74c3c", ls=":", alpha=0.6, label="floor (0.75)")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=8, facecolor="#1a1a2e", edgecolor="#333355", labelcolor="#ccccdd")
axes[0].set_title("Model Performance", fontsize=11)

# Panel 2: PSI drift
axes[1].plot(df["timestamp"], df["drift_score"], color="#4fc3f7", lw=1.5, label="PSI drift")
axes[1].fill_between(df["timestamp"], df["drift_score"], alpha=0.15, color="#4fc3f7")
axes[1].axhline(0.10, color="#ffcc02", ls=":", alpha=0.6, label="moderate (0.10)")
axes[1].axhline(0.20, color="#e74c3c", ls=":", alpha=0.6, label="reject   (0.20)")
axes[1].set_ylabel("PSI")
axes[1].legend(fontsize=8, facecolor="#1a1a2e", edgecolor="#333355", labelcolor="#ccccdd")
axes[1].set_title("Feature Drift (PSI)", fontsize=11)

# Panel 3: decisions
action_rank = {"none": 0, "retrain": 1, "rollback": 2, "reject": 3, "promote": 4}
events = df[df["action"] != "none"]
for _, row in events.iterrows():
    c = ACTION_COLOURS.get(row["action"], "#95a5a6")
    r = action_rank.get(row["action"], 0)
    axes[2].scatter(row["timestamp"], r, color=c, s=60, zorder=3)
axes[2].set_yticks(list(action_rank.values()))
axes[2].set_yticklabels(list(action_rank.keys()), fontsize=9)
axes[2].set_xlabel("Time")
axes[2].set_title("Decision Events", fontsize=11)
patchlist = [mpatches.Patch(color=c, label=a) for a, c in ACTION_COLOURS.items()]
axes[2].legend(handles=patchlist, fontsize=8, ncol=4,
               facecolor="#1a1a2e", edgecolor="#333355", labelcolor="#ccccdd")

plt.tight_layout()
fig.suptitle("model_monitor -  Live simulation data", color="#eeeeff", fontsize=13, y=1.01)
plt.show()

## Decision audit log

In [ ]:
decisions = df[df["action"] != "none"][["timestamp", "action", "reason"]].copy()
decisions["timestamp"] = decisions["timestamp"].dt.strftime("%H:%M:%S")
if len(decisions) == 0:
    print("No non-trivial decisions recorded. Try a longer simulation.")
else:
    print(decisions.to_string(index=False))

## PSI drift distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("#0f1117")
ax.set_facecolor("#1a1a2e")
ax.spines[:].set_color("#333355")

ax.hist(df["drift_score"], bins=30, color="#4fc3f7", alpha=0.8, edgecolor="#0f1117")
ax.axvline(0.10, color="#ffcc02", ls="--", lw=1.5, label="moderate (0.10)")
ax.axvline(0.20, color="#e74c3c", ls="--", lw=1.5, label="severe / reject (0.20)")
ax.set_xlabel("PSI drift score", color="#ccccdd")
ax.set_ylabel("Batch count", color="#ccccdd")
ax.legend(fontsize=9, facecolor="#1a1a2e", edgecolor="#333355", labelcolor="#ccccdd")
ax.set_title("Distribution of PSI drift across all batches", color="#eeeeff", fontsize=11)
fig.patch.set_facecolor("#0f1117")
plt.tight_layout()
plt.show()

pct_stable   = (df["drift_score"] <  0.10).mean() * 100
pct_moderate = ((df["drift_score"] >= 0.10) & (df["drift_score"] < 0.20)).mean() * 100
pct_severe   = (df["drift_score"] >= 0.20).mean() * 100
print(f"Stable   (PSI < 0.10):  {pct_stable:.1f}% of batches")
print(f"Moderate (0.10–0.20):   {pct_moderate:.1f}% of batches")
print(f"Severe   (PSI ≥ 0.20):  {pct_severe:.1f}% of batches")